# IEEE-CIS Fraud Detection: Dataset Audit

## Objective

Understand the structure, scale, target distribution, temporal characteristics,
missingness, and relationships within the IEEE-CIS Fraud Detection dataset
before developing any predictive model.

## Questions

1. What does one observation represent?
2. How large are the transaction and identity datasets?
3. How is the transaction table related to the identity table?
4. How prevalent is fraud?
5. How are transactions distributed over time?
6. How much missing information exists?
7. Which feature groups are available?
8. What potential sources of leakage exist?

## Principle

No modeling decisions will be made before the dataset has been properly audited.

In [1]:
%load_ext sql

In [2]:
import duckdb

conn = duckdb.connect()
%sql conn --alias duckdb

In [11]:
%%sql

CREATE OR REPLACE VIEW train_transaction AS SELECT * FROM read_csv_auto('../data/raw/train_transaction.csv');

CREATE OR REPLACE VIEW train_identity AS SELECT * FROM read_csv_auto('../data/raw/train_identity.csv');

CREATE OR REPLACE VIEW test_transaction AS SELECT * FROM read_csv_auto('../data/raw/test_transaction.csv');

CREATE OR REPLACE VIEW test_identity AS SELECT * FROM read_csv_auto('../data/raw/test_identity.csv');

CREATE OR REPLACE VIEW sample_submission AS SELECT * FROM read_csv_auto('../data/raw/sample_submission.csv');

Running query in 'duckdb'

Count


In [12]:
%%sql

SHOW TABLES;

Running query in 'duckdb'

name
sample_submission
test_identity
test_transaction
train_identity
train_transaction


In [13]:
%%sql

Select *
FROM sample_submission
LIMIT 5;

Running query in 'duckdb'

TransactionID,isFraud
3663549,0.5
3663550,0.5
3663551,0.5
3663552,0.5
3663553,0.5


In [14]:
%%sql

SELECT *
FROM test_identity
LIMIT 5;

Running query in 'duckdb'

TransactionID,id-01,id-02,id-03,id-04,id-05,id-06,id-07,id-08,id-09,id-10,id-11,id-12,id-13,id-14,id-15,id-16,id-17,id-18,id-19,id-20,id-21,id-22,id-23,id-24,id-25,id-26,id-27,id-28,id-29,id-30,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
3663586,-45.0,280290.0,None,None,0.0,0.0,None,None,None,None,100.0,NotFound,27.0,None,New,NotFound,225.0,15.0,427.0,563.0,None,None,None,None,None,None,None,New,NotFound,None,chrome 67.0 for android,None,None,None,False,False,True,False,mobile,MYA-L13 Build/HUAWEIMYA-L13
3663588,0.0,3579.0,0.0,0.0,0.0,0.0,None,None,0.0,0.0,100.0,Found,None,-300.0,Found,Found,166.0,None,542.0,368.0,None,None,None,None,None,None,None,Found,Found,Android 6.0.1,chrome 67.0 for android,24.0,1280x720,match_status:2,True,False,True,True,mobile,LGLS676 Build/MXB48T
3663597,-5.0,185210.0,None,None,1.0,0.0,None,None,None,None,100.0,NotFound,52.0,-360.0,New,NotFound,225.0,None,271.0,507.0,None,None,None,None,None,None,None,New,NotFound,None,ie 11.0 for tablet,None,None,None,False,True,True,False,desktop,Trident/7.0
3663601,-45.0,252944.0,0.0,0.0,0.0,0.0,None,None,0.0,0.0,100.0,NotFound,27.0,None,Found,Found,225.0,15.0,427.0,563.0,None,None,None,None,None,None,None,Found,Found,None,chrome 67.0 for android,None,None,None,False,False,True,False,mobile,MYA-L13 Build/HUAWEIMYA-L13
3663602,-95.0,328680.0,None,None,7.0,-33.0,None,None,None,None,100.0,NotFound,27.0,None,New,NotFound,225.0,15.0,567.0,507.0,None,None,None,None,None,None,None,New,NotFound,None,chrome 67.0 for android,None,None,None,False,False,True,False,mobile,SM-G9650 Build/R16NW


In [10]:
%%sql

SELECT *
FROM train_transaction
LIMIT 5;

Running query in 'duckdb'

TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,M5,M6,M7,M8,M9,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,V29,V30,V31,V32,V33,V34,V35,V36,V37,V38,V39,V40,V41,V42,V43,V44,V45,V46,V47,V48,V49,V50,V51,V52,V53,V54,V55,V56,V57,V58,V59,V60,V61,V62,V63,V64,V65,V66,V67,V68,V69,V70,V71,V72,V73,V74,V75,V76,V77,V78,V79,V80,V81,V82,V83,V84,V85,V86,V87,V88,V89,V90,V91,V92,V93,V94,V95,V96,V97,V98,V99,V100,V101,V102,V103,V104,V105,V106,V107,V108,V109,V110,V111,V112,V113,V114,V115,V116,V117,V118,V119,V120,V121,V122,V123,V124,V125,V126,V127,V128,V129,V130,V131,V132,V133,V134,V135,V136,V137,V138,V139,V140,V141,V142,V143,V144,V145,V146,V147,V148,V149,V150,V151,V152,V153,V154,V155,V156,V157,V158,V159,V160,V161,V162,V163,V164,V165,V166,V167,V168,V169,V170,V171,V172,V173,V174,V175,V176,V177,V178,V179,V180,V181,V182,V183,V184,V185,V186,V187,V188,V189,V190,V191,V192,V193,V194,V195,V196,V197,V198,V199,V200,V201,V202,V203,V204,V205,V206,V207,V208,V209,V210,V211,V212,V213,V214,V215,V216,V217,V218,V219,V220,V221,V222,V223,V224,V225,V226,V227,V228,V229,V230,V231,V232,V233,V234,V235,V236,V237,V238,V239,V240,V241,V242,V243,V244,V245,V246,V247,V248,V249,V250,V251,V252,V253,V254,V255,V256,V257,V258,V259,V260,V261,V262,V263,V264,V265,V266,V267,V268,V269,V270,V271,V272,V273,V274,V275,V276,V277,V278,V279,V280,V281,V282,V283,V284,V285,V286,V287,V288,V289,V290,V291,V292,V293,V294,V295,V296,V297,V298,V299,V300,V301,V302,V303,V304,V305,V306,V307,V308,V309,V310,V311,V312,V313,V314,V315,V316,V317,V318,V319,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
2987000,0,86400,68.5,W,13926,None,150.0,discover,142.0,credit,315.0,87.0,19.0,None,None,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,14.0,None,13.0,None,None,None,None,None,None,13.0,13.0,None,None,None,0.0,True,True,True,M2,False,True,None,None,None,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,None,None,gmail.com,None,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,None,None,0.0,None,None,None,None,None,0.0,None,None,None,None,0.0,None,None,None,M0,True,True,None,None,None,None,None,None,None,None,None,None,None,None,None,None,0.0,0.

In [16]:
%%sql
DESCRIBE train_transaction;

Running query in 'duckdb'

column_name,column_type,null,key,default,extra
TransactionID,BIGINT,YES,None,None,None
isFraud,BIGINT,YES,None,None,None
TransactionDT,BIGINT,YES,None,None,None
TransactionAmt,DOUBLE,YES,None,None,None
ProductCD,VARCHAR,YES,None,None,None
card1,BIGINT,YES,None,None,None
card2,DOUBLE,YES,None,None,None
card3,DOUBLE,YES,None,None,None
card4,VARCHAR,YES,None,None,None
card5,DOUBLE,YES,None,None,None


In [ ]:
%%sql

SELECT COUNT(*) AS total_transactions
FROM 